# Deep Learning for Image Inpainting

Image inpainting reconstructs missing regions of an image using the visible surrounding content. This project will compare a lightweight **U-Net**, a lightweight **Vision Transformer (ViT)**, and their **weighted ensemble**.

## Phase 1: Dataset preparation

This phase loads original images, creates reproducible data splits, and checks the data. Run the cells in order in Google Colab. Masking and models will be added in later phases.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
IMAGE_SIZE = 128
BATCH_SIZE = 32
NUM_WORKERS = 2
TRAIN_FRACTION = 0.8
# Download into Colab's temporary runtime storage when this cell is run there.
DATA_ROOT = Path('/content/data') if Path('/content').is_dir() else Path('data')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Dataset location: {DATA_ROOT.resolve()}')

## Oxford-IIIT Pet Dataset and splits

Oxford-IIIT Pet contains photographs of cats and dogs from 37 pet categories. We use the photographs for reconstruction; category labels returned by torchvision are not needed for inpainting.

The **official test split remains untouched** and reserved for final evaluation. We split only the official **trainval** split into **80% training / 20% validation**, using a fixed seed. Both future models will use these same subsets.

Images are loaded as RGB by torchvision, resized to **128 × 128**, and converted to floating-point tensors in **[0, 1]**. No ImageNet mean/std normalization is applied. The first run downloads the dataset into the runtime's data folder and may take a few minutes.

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

trainval_dataset = datasets.OxfordIIITPet(
    root=DATA_ROOT,
    split='trainval',
    target_types='category',
    transform=image_transform,
    download=True,
)
test_dataset = datasets.OxfordIIITPet(
    root=DATA_ROOT,
    split='test',
    target_types='category',
    transform=image_transform,
    download=True,
)

train_size = int(TRAIN_FRACTION * len(trainval_dataset))
val_size = len(trainval_dataset) - train_size
train_dataset, val_dataset = random_split(
    trainval_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

In [ ]:
print(f'Official trainval: {len(trainval_dataset):,}')
print(f'Training:         {len(train_dataset):,}')
print(f'Validation:       {len(val_dataset):,}')
print(f'Official test:     {len(test_dataset):,}')

# Check that training and validation partition trainval without overlap.
train_indices = set(train_dataset.indices)
val_indices = set(val_dataset.indices)
assert len(train_indices) == train_size > 0
assert len(val_indices) == val_size > 0
assert train_indices.isdisjoint(val_indices)
assert train_indices | val_indices == set(range(len(trainval_dataset)))
assert test_dataset is not trainval_dataset
assert len(test_dataset) > 0

# Verify that the same seed reproduces exactly the same split.
expected_indices = torch.randperm(
    len(trainval_dataset), generator=torch.Generator().manual_seed(SEED)
).tolist()
assert train_dataset.indices == expected_indices[:train_size]
assert val_dataset.indices == expected_indices[train_size:]

# Inspect a training batch only; do not iterate over the test set.
images, _ = next(iter(train_loader))
print(f'Batch shape: {list(images.shape)}')
print(f'Pixel range: [{images.min().item():.3f}, {images.max().item():.3f}]')
assert images.shape == (min(BATCH_SIZE, train_size), 3, IMAGE_SIZE, IMAGE_SIZE)
assert images.dtype == torch.float32
assert torch.isfinite(images).all().item()
assert 0.0 <= images.min().item() <= images.max().item() <= 1.0
print('All split and training-batch sanity checks passed.')

In [ ]:
# Display eight original training images from the checked batch.
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for index, ax in enumerate(axes.flat):
    ax.imshow(images[index].permute(1, 2, 0).numpy())
    ax.set_title(f'Training image {index + 1}')
    ax.axis('off')
fig.suptitle('Original training images (128 x 128)')
plt.tight_layout()
plt.show()